# Scenario 4: Prompt Engineering Lab with Galileo Experiments

## What You'll Learn

In this notebook you will:

- **Create evaluation datasets** — collections of test questions with expected answers
- **Build prompt templates** — reusable prompts with `{{input}}` placeholders
- **Run experiments** to compare different prompts and models side by side
- **Use expression metrics** (BLEU, ROUGE, tone) to measure writing style and reference similarity
- **Check model confidence** with perplexity and uncertainty scores

## What Are Experiments?

When you build an LLM-powered feature, you inevitably ask: *"Is this prompt good enough?"* Testing one question at a time in a chat window does not scale. **Experiments** are Galileo's systematic way to answer that question with data.

An experiment takes a **prompt template** (or a Python function), runs it across an **entire dataset** of test inputs, and **automatically scores every response** using the metrics you choose. The results are saved so you can compare multiple experiment runs side by side.

## When to Use Experiments

| Situation | Example |
|---|---|
| Comparing **prompt versions** before shipping | "Does the v2 system prompt improve accuracy?" |
| Comparing **model settings** (temperature, max_tokens) | "Is temperature 0.3 or 0.8 better for factual Q&A?" |
| Checking quality against **reference answers** | "How close are the responses to our gold-standard answers?" |
| Evaluating **output characteristics** like tone | "Are responses consistently professional and neutral?" |

## Key Concepts

| Concept | What It Means |
|---|---|
| **Dataset** | A collection of input/output test pairs. The "input" is the question; the "output" is the expected (reference) answer. |
| **Prompt Template** | A reusable prompt with `{{input}}` placeholders that get filled from each dataset row. |
| **Experiment Run** | One execution of a template (or function) over a dataset. Each run produces per-row scores. |
| **PromptRunSettings** | Model configuration for an experiment — `model_alias`, `temperature`, `max_tokens`, etc. |
| **Expression Metrics** | BLEU, ROUGE, tone — metrics that measure writing style and similarity to reference text. |
| **Confidence Metrics** | Perplexity and uncertainty — how "sure" the model is about its output. |

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY` set
- The `galileo` Python SDK installed (`uv sync` from the repo root)
- An OpenAI integration configured in your Galileo console (the `model_alias` in experiments must match)

### Step 0: Load Environment Variables

The cell below loads your `.env` file, which must contain `GALILEO_API_KEY` and `OPENAI_API_KEY`. These credentials let the SDK talk to Galileo's servers and call the OpenAI API during experiments.

In [2]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


### Step 0b: Import Libraries

Here is what the key imports do:

- **`GalileoMetrics`** — an enum of all built-in metrics you can attach to an experiment (correctness, BLEU, ROUGE, tone, etc.)
- **`galileo_context`** — the main entry point for initializing a Galileo project/log-stream session
- **`create_dataset` / `get_dataset` / `delete_dataset`** — CRUD helpers for managing evaluation datasets
- **`run_experiment`** — the core function that executes an experiment run over a dataset
- **`PromptRunSettings`** — a dataclass to configure model parameters (model_alias, temperature, max_tokens) for an experiment
- **`PROJECT_NAME`, `DATASET_NAME`, etc.** — constants so every cell in this notebook points at the same Galileo project and dataset

In [ ]:
from galileo import GalileoMetrics, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset, get_dataset
from galileo.experiments import PromptRunSettings, run_experiment
from galileo.log_streams import create_log_stream, get_log_stream
from galileo.projects import delete_project

PROJECT_NAME = 'GalileoEval_S4_Experiments'
LOG_STREAM_NAME = 'experiment-lab'
DATASET_NAME = 'prompt-lab-dataset'
BASELINE_TEMPLATE_NAME = 'concise-scientific-prompt-template'
dataset_id = None

## 1. Initialize Galileo

This creates (or connects to) a Galileo **project** and **log stream**. Think of the project as a folder that holds all your datasets, prompt templates, and experiment runs. The log stream is where real-time traces go. For this notebook the important part is the project — experiments will be saved there.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

## 2. Create a Dataset

A **dataset** is your test suite — a collection of question/answer pairs that experiments run against.

- The **"input"** column contains the questions that get fed into your prompt template's `{{input}}` placeholder.
- The **"output"** column is the **reference (expected) answer**. Metrics like `ground_truth_adherence`, BLEU, and ROUGE compare the model's response against this reference.
- `get_dataset` checks if the dataset already exists before creating it, so this cell is **idempotent** — safe to re-run without duplicating data.
- Datasets live inside the project and can be **reused across many experiments**. You create the dataset once, then run different prompts and models against it.

In [ ]:
ds = get_dataset(name=DATASET_NAME)
if not ds:
    ds = create_dataset(
        name=DATASET_NAME,
        content=[
            {'input': 'Explain photosynthesis in one sentence.', 'output': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.'},
            {'input': 'What causes rain?', 'output': 'Rain occurs when water vapor condenses into droplets that fall when heavy enough.'},
            {'input': 'Why is the sky blue?', 'output': 'The sky appears blue because shorter blue wavelengths scatter more than other colors.'},
            {'input': 'How do vaccines work?', 'output': 'Vaccines train the immune system to recognize and fight pathogens.'},
        ],
        project_name=PROJECT_NAME,
    )
dataset_id = ds.id
str(dataset_id)

## 3. Add More Dataset Rows

You can **expand datasets incrementally** with `add_rows()` — no need to recreate the whole dataset.

More diverse test cases lead to more reliable experiment results. In practice you would build datasets from:

- **Real user questions** collected from production logs
- **Edge cases** that have caused problems before
- **Known failure modes** where your prompt tends to struggle

The two rows added here cover different topics (physics, networking) to broaden coverage.

In [7]:
ds = get_dataset(id=str(dataset_id))
ds.add_rows([
    {'input': 'What is gravity?', 'output': 'Gravity is the force of attraction between objects with mass.'},
    {'input': 'How does WiFi work?', 'output': 'WiFi uses radio waves to transmit data wirelessly between devices and a router.'},
])
'Added 2 rows'


'Added 2 rows'

## 4. Create a Prompt Template and Run the Baseline Experiment

This is the most common experiment workflow and the most important cell in the notebook.

**Prompt Templates** are reusable and versioned inside Galileo. The `{{input}}` placeholder gets automatically filled with each row's "input" value from the dataset.

This run is your **BASELINE** — all future experiments compare against it. The metrics chosen here score every dataset row:

- **`correctness`** — Is the answer factually correct?
- **`instruction_adherence`** — Did the model follow the system prompt instructions (e.g., "one sentence")?
- **`ground_truth_adherence`** — How well does the response match the reference answer in the dataset's "output" column?

**`PromptRunSettings`** configures the model for this experiment:
- `model_alias` must match one of the integrations configured in your Galileo console (e.g., "GPT-4o mini")
- `temperature=0.3` keeps responses relatively deterministic
- `max_tokens=100` caps response length

**What to check in the UI:** Open the **Experiments** tab in your Galileo console, find this run, and review per-row scores. This baseline gives you a reference point for every experiment that follows.

In [ ]:
from galileo import Message, MessageRole
from galileo.prompts import GlobalPromptTemplates

templates = GlobalPromptTemplates()
prompt_template = templates.get(name=BASELINE_TEMPLATE_NAME)
if prompt_template is None:
    prompt_template = templates.create(
        name=BASELINE_TEMPLATE_NAME,
        template=[
            Message(role=MessageRole.system, content='Answer in exactly one sentence. Be precise and scientific.'),
            Message(role=MessageRole.user, content='{{input}}'),
        ],
        project_name=PROJECT_NAME,
    )

baseline_run = run_experiment(
    experiment_name='concise-scientific-prompt',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=100),
    project=PROJECT_NAME,
)
baseline_run['message']


'Concise prompt experiment complete'

## 5. Run a Code-Based Experiment

Instead of a prompt template, you can pass a **Python function** to `run_experiment`. The function receives `input_text` (a string from each dataset row) and returns a string.

**When to use this:**
- Testing **post-processing logic** (e.g., reformatting model output)
- Testing **custom generation pipelines** that do more than a single LLM call
- Comparing a **hard-coded heuristic** against an LLM-based approach

The function below is intentionally simple (it does not call an LLM). This lets you see how Galileo scores a deterministic transformation compared to the LLM baseline.

**What to check in the UI:** In the Experiments tab, compare this run against the baseline from Step 4. Does the code-based approach score differently on correctness or instruction adherence?

In [11]:
def eli5_runner(input_text: str) -> str:
    return f"Imagine you're 5 years old. {input_text} works like a simple science trick."

function_run = run_experiment(
    experiment_name='eli5-function-experiment',
    dataset=DATASET_NAME,
    function=eli5_runner,
    metrics=[GalileoMetrics.correctness, GalileoMetrics.instruction_adherence],
    project=PROJECT_NAME,
)
function_run['message']


'Code-based experiment complete'

## 6. Run an Expression Metrics Experiment

This experiment uses a different set of metrics that focus on **writing style** and **reference similarity** rather than factual correctness.

Here is what each metric measures:

| Metric | What It Does |
|---|---|
| **BLEU** | Measures **n-gram overlap** between the generated response and the reference answer. Borrowed from machine translation — higher score means the response uses similar words and phrases as the reference. |
| **ROUGE** | Measures **recall of reference n-grams** in the generated text. Answers the question: "How much of the reference answer's content appears in the response?" |
| **input_tone** | Classifies the **emotional tone of the prompt** (e.g., neutral, curious, formal). Useful for understanding how input phrasing affects results. |
| **output_tone** | Classifies the **emotional tone of the response** (e.g., informative, casual, authoritative). Useful for ensuring consistent voice. |

**When to use these metrics:** When you care about **writing style consistency**, not just factual accuracy. For example, a customer-support bot needs to sound professional and match approved answer templates — BLEU/ROUGE and tone metrics help you verify that at scale.

**What to check in the UI:** Open this run in Experiments and inspect BLEU/ROUGE scores alongside tone classifications. Compare against the baseline to see whether a prompt that scores well on correctness also produces the writing style you want.

In [13]:
expression_run = run_experiment(
    experiment_name='expression-metrics-comparison',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.bleu,
        GalileoMetrics.rouge,
        GalileoMetrics.input_tone,
        GalileoMetrics.output_tone,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=100),
    project=PROJECT_NAME,
)
expression_run['message']


[]

## 7. Check Model Confidence

Galileo can measure how "sure" the model is about its output. These metrics help identify responses where the model is guessing rather than confidently answering.

| Metric | What It Measures |
|---|---|
| **Perplexity** | How surprised the model is by its own output. Lower perplexity = more confident. A response the model finds "natural" has low perplexity. |
| **Uncertainty** | The model's self-reported confidence. Higher uncertainty means the model is less sure about its answer. |

**When to use these:** Confidence metrics are especially useful for:
- **Flagging unreliable responses** — High uncertainty + low correctness = the model doesn't know and is making things up
- **A/B testing temperature settings** — Higher temperature increases creativity but may also increase uncertainty
- **Routing decisions** — Send low-confidence responses to human review instead of showing them to users

In [14]:
confidence_run = run_experiment(
    experiment_name='temperature-confidence-comparison',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.prompt_perplexity,
        GalileoMetrics.uncertainty,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.8, max_tokens=150),
    project=PROJECT_NAME,
)
confidence_run['message']


[]

## 8. Fine-Tuned Metrics (Enterprise Feature)

**Fine-tuned metrics** let you create **domain-specific evaluation models** that go beyond generic quality scores. Instead of asking "is this response correct?", you can train a metric to ask "does this medical response follow our clinical guidelines?"

### How Fine-Tuned Metrics Work

1. **Collect labeled data** — Export annotated traces where human reviewers scored quality on your custom criteria
2. **Train a custom scorer** — Galileo fine-tunes a small evaluation model on your labeled data
3. **Deploy as a metric** — The fine-tuned scorer runs automatically on new traces, just like built-in metrics

### When to Use Fine-Tuned Metrics

| Use Case | Generic Metric Limitation | Fine-Tuned Solution |
|----------|---------------------------|---------------------|
| Medical advice quality | "Correctness" doesn't know clinical guidelines | Train on expert-annotated medical traces |
| Legal document review | "Instruction adherence" misses legal nuance | Train on lawyer-reviewed outputs |
| Brand voice consistency | "Tone" is too generic | Train on brand-approved vs. rejected responses |

### Requirements

- **Enterprise plan** required
- A minimum of ~100 labeled examples recommended for good performance
- Labels can come from human annotations (Scenario 7) or programmatic rules

> **Note:** Fine-tuned metrics are an Enterprise feature. Contact your Galileo team for access. For most use cases, the combination of built-in metrics + custom code metrics (Scenario 6) provides excellent coverage.

## 9. Export Experiment Results (Console UI)

The Galileo Console provides rich tools for **comparing and exporting** experiment results.

### Experiment Comparison View

1. Open your **project** in the Galileo Console
2. Go to the **Experiments** tab
3. Select two or more experiment runs to compare
4. Galileo shows a **side-by-side comparison** with per-row scores for each run

### Export Options

- **CSV Export** — Download all experiment results (inputs, outputs, metric scores) as a CSV file
- **Charts** — View metric distributions, box plots, and score histograms directly in the console
- **Per-row comparison** — Click any row to see the full prompt, response, and metric breakdown for each run

### What to Look For

| Comparison | What It Tells You |
|------------|-------------------|
| Baseline vs. high-temperature run | Does randomness help or hurt quality? |
| Prompt v1 vs. v2 | Did your prompt revision actually improve scores? |
| Code function vs. LLM template | Is the LLM better than your heuristic? |

### Pro Tip

Export results from multiple experiments into one spreadsheet. Add a "run_name" column and use pivot tables to compare metrics across all runs at once.

## 10. Re-Run Experiments with Modified Parameters

One of the most valuable experiment workflows is **re-running the same prompt template with different model settings**. This lets you systematically explore how parameters like `temperature` and `max_tokens` affect output quality.

### What We're Testing

We'll re-run our baseline prompt with `temperature=0.9` (more creative/random) and `max_tokens=200` (allow longer responses). Compare this against the baseline (`temperature=0.3`, `max_tokens=100`) to see how these changes affect correctness and instruction adherence.

In [ ]:
rerun = run_experiment(
    experiment_name='high-temp-rerun',
    dataset=DATASET_NAME,
    prompt_template=prompt_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.9, max_tokens=200),
    project=PROJECT_NAME,
)
rerun['message']

## 11. Monitor Long-Running Experiment Jobs

When you run experiments with large datasets or multiple metrics, the job can take a while. Galileo provides **job monitoring** in the Console so you can track progress.

### Job States

| State | Meaning |
|-------|---------|
| **Pending** | Job is queued and waiting for resources |
| **Running** | Job is actively processing dataset rows |
| **Completed** | All rows processed and scored successfully |
| **Failed** | An error occurred (check job details for the error message) |

### How to Monitor

1. Open your **project** in the Galileo Console
2. Go to the **Jobs** tab (or check the experiment's status indicator)
3. You'll see all experiment runs with their current state, progress percentage, and estimated time remaining
4. Click a job to see detailed logs and per-row progress

### SDK-Side Behavior

When you call `run_experiment()`, the SDK:
1. Submits the job to the Galileo backend
2. Polls for completion (you'll see progress in the notebook output)
3. Returns the results once the job finishes

For very large datasets, you can also submit the job and check results later in the Console — the job runs server-side regardless of whether your notebook is still connected.

### Pro Tip

If an experiment job fails partway through, check the Console for the specific row that caused the error. Often it's a single malformed input. Fix it in your dataset and re-run.

## 12. Clean Up the Dataset and Project

In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

try:
    delete_project(name=PROJECT_NAME)
except Exception:
    pass
